# OptiClean – Stage 1: EDA & Demand Prediction Model
**Course:** Decision Analytics in the AI Era (2025/2026) – ESSEC | **Group M**

---
This notebook covers the Stage 1 prediction pipeline:
1. Brief EDA to understand demand drivers and justify feature choices
2. Regression model comparison and selection
3. Generation of the **200 × 8 demand matrix** handed to the optimisation model


## 0. Imports & Data Loading

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
from scipy.stats import ttest_ind
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error

hist  = pd.read_excel("data/historic_clients.xlsx")
pot   = pd.read_excel("data/potential_clients.xlsx")
sites = pd.read_excel("data/candidate_sites.xlsx")
sites.index = [f"Site {j+1}" for j in range(len(sites))]

print(f"Historic clients  : {hist.shape}  (training data)")
print(f"Potential clients : {pot.shape}  (Valmontier – no demand observed)")
print(f"Candidate sites   : {sites.shape}")
print()
# What's available for new clients vs not
only_historic = sorted(set(hist.columns) - set(pot.columns))
print("Features ONLY in historic data (cannot use in prediction model):")
for f in only_historic:
    print(f"  ✗  {f}")

## 1. Exploratory Data Analysis

### 1.1 Key Statistics


In [ ]:
print(hist[['employes','surface','firm_age','monthly_volume',
             'purchase_staff','dist_retail_point','demand']].describe().round(2).to_string())
print()
print("Loyalty split:", hist['loyal'].value_counts().rename({0:'Non-loyal',1:'Loyal'}).to_dict())
print()
# Demand by loyalty
g = hist.groupby('loyal')['demand'].agg(['mean','std','median'])
g.index = ['Non-loyal','Loyal']
print("Demand by loyalty status:")
print(g.round(2))
_, p = ttest_ind(hist[hist['loyal']==1]['demand'], hist[hist['loyal']==0]['demand'])
print(f"\nt-test p-value: {p:.2e}  → difference is statistically significant")

### 1.2 EDA Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) Correlations with demand
num_cols = ['surface','employes','purchase_staff','monthly_volume',
            'firm_age','dist_retail_point','loyal','demand']
corr_d = hist[num_cols].corr()['demand'].drop('demand').sort_values(ascending=False)
bar_colors = ['#1a9641' if v > 0 else '#d73027' for v in corr_d]
bars = axes[0].barh(corr_d.index, corr_d.values, color=bar_colors, edgecolor='black', lw=0.5)
axes[0].axvline(0, color='black', lw=0.8)
for bar, v in zip(bars, corr_d.values):
    axes[0].text(v + (0.01 if v>=0 else -0.01), bar.get_y()+bar.get_height()/2,
                 f'{v:.3f}', va='center', ha='left' if v>=0 else 'right', fontsize=9)
axes[0].set_title("(a) Feature Correlations with Weekly Demand", fontsize=11)
axes[0].set_xlabel("Pearson r"); axes[0].set_xlim(-0.55, 0.95)

# (b) Demand vs Distance coloured by loyalty — motivates keeping distance as a feature
for lv, col, lbl in [(0,'#2c7bb6','Non-loyal (n=395)'),(1,'#d7191c','Loyal (n=105)')]:
    g = hist[hist['loyal']==lv]
    axes[1].scatter(g['dist_retail_point'], g['demand'], alpha=0.35, s=18, color=col, label=lbl)
    z = np.polyfit(g['dist_retail_point'], g['demand'], 1)
    xr = np.linspace(g['dist_retail_point'].min(), g['dist_retail_point'].max(), 50)
    axes[1].plot(xr, np.polyval(z,xr), color=col, lw=2)
axes[1].set_xlabel("Distance to Retail Point"); axes[1].set_ylabel("Weekly Demand")
axes[1].set_title("(b) Demand vs Distance (per loyalty group)", fontsize=11)
axes[1].legend(fontsize=8)

plt.suptitle("Figure 1: Key EDA Findings", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig("plots_clean/fig1_eda.png", dpi=150, bbox_inches='tight'); plt.show()

**Takeaways:**
- **Surface (r = 0.80)** is the dominant predictor — larger firms order significantly more.
- **Distance (r = −0.37)** has a clear negative effect and must be kept in the model; its value will be pre-computed per client–site pair (see Section 3).
- **Loyal clients** have 72% higher mean demand (84.3 vs 49.1) and are on average 13 units closer to their retail point. This structural difference motivates Stage 2's segmented approach.
- `purchased_products` (r ≈ 0.04) is excluded: near-zero predictive value *and* unavailable for new clients.


## 2. Feature Selection

| Feature | Available for new clients? | Include? | Reason |
|---|---|---|---|
| `surface` | ✅ | ✅ | Strongest predictor (r = 0.80) |
| `employes` | ✅ | ✅ | Strong predictor (r = 0.41) |
| `purchase_staff` | ✅ | ✅ | Moderate positive effect |
| `firm_age` | ✅ | ✅ | Included (small but non-zero effect) |
| `monthly_volume` | ✅ | ✅ | Included |
| `dist_retail_point` | ❌ not observed | ✅ computed | Pre-computed as Euclidean distance to each candidate site |
| `purchased_products` | ❌ not in dataset | ❌ | Unavailable + near-zero correlation |
| `loyal` | ❌ unknown | ❌ S1 / ✅ S2 | Unknown for new clients; predicted via classifier in Stage 2 |


In [ ]:
FEATURES = ['employes','surface','firm_age','monthly_volume','purchase_staff','dist_retail_point']
print("Final feature set:", FEATURES)

## 3. Demand Prediction Model

In [ ]:
X = hist[FEATURES].values
y = hist['demand'].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

candidate_models = {
    "OLS Linear"      : Pipeline([('s', StandardScaler()), ('r', LinearRegression())]),
    "Ridge (α=1)"     : Pipeline([('s', StandardScaler()), ('r', Ridge(alpha=1.0))]),
    "Random Forest"   : RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, random_state=42),
}

print(f"{'Model':22s}  {'CV R²':>15}  {'CV RMSE':>10}")
print("-" * 55)
cv_results = {}
for name, m in candidate_models.items():
    r2   = cross_val_score(m, X, y, cv=kf, scoring='r2')
    rmse = np.sqrt(-cross_val_score(m, X, y, cv=kf, scoring='neg_mean_squared_error'))
    cv_results[name] = {'r2': r2.mean(), 'r2_std': r2.std(), 'rmse': rmse.mean()}
    print(f"{name:22s}  {r2.mean():.4f} ±{r2.std():.4f}  {rmse.mean():10.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_m = ['#e41a1c','#377eb8','#4daf4a','#984ea3']
names = list(cv_results.keys())

bars = axes[0].bar(names, [cv_results[n]['r2'] for n in names],
                   yerr=[cv_results[n]['r2_std'] for n in names],
                   capsize=5, color=colors_m, edgecolor='black', lw=0.7)
axes[0].set_ylim(0.84, 0.96); axes[0].set_ylabel("CV R²")
axes[0].set_title("(a) 5-fold CV R²", fontsize=11)
axes[0].set_xticklabels(names, rotation=12, ha='right')
for bar, v in zip(bars, [cv_results[n]['r2'] for n in names]):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.001, f'{v:.4f}',
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

# Fit OLS on full data for diagnostics
ols = Pipeline([('s',StandardScaler()),('r',LinearRegression())])
ols.fit(X, y)
y_pred = ols.predict(X)
resid  = y - y_pred

axes[1].scatter(y_pred, y, alpha=0.4, s=18, color='steelblue')
lim = max(y.max(), y_pred.max())*1.05
axes[1].plot([0,lim],[0,lim],'r--',lw=1.5,label='Perfect fit')
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
axes[1].set_title(f"(b) Actual vs Predicted  (R²={r2_score(y,y_pred):.4f})", fontsize=11)
axes[1].legend(fontsize=8)

axes[2].scatter(y_pred, resid, alpha=0.4, s=18, color='purple')
axes[2].axhline(0, color='red', lw=1.5)
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("Residual")
axes[2].set_title("(c) Residuals vs Fitted", fontsize=11)

plt.suptitle("Figure 2: Model Comparison & OLS Diagnostics", fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig("plots_clean/fig2_model.png", dpi=150, bbox_inches='tight'); plt.show()

# Coefficients
coefs = pd.Series(ols.named_steps['r'].coef_, index=FEATURES)
print("\nOLS standardised coefficients:")
for f, c in coefs.sort_values(key=abs, ascending=False).items():
    print(f"  {f:25s}: {c:+.4f}")

**Model selection: OLS Linear Regression**

OLS achieves the best CV R² (0.9194) with the lowest RMSE (10.23). The negligible gap with Ridge confirms no multicollinearity. Tree models underperform — confirming the relationships are predominantly linear, consistent with the high Pearson correlations.

OLS is also preferred because it will naturally produce **negative predictions** for very distant client–site pairs (linear extrapolation beyond the training distance range of 1.7–63.4 units), which the case explicitly requires clamping to zero. This cannot be observed with tree models.


## 4. Demand Matrix Generation (200 × 8)

### The circular dependency
Distance is a key feature in the model, but for potential clients it depends on which store is eventually opened. We resolve this by pre-computing the Euclidean distance between **every client–site pair** and running the model for each combination — producing a full demand matrix before any optimisation takes place.

$$\hat{d}_{i,j} = \max\bigl(0,\ \text{OLS}(\mathbf{x}_i,\ d_{i,j})\bigr) \quad \forall\, i \in \{1{,}\ldots,200\},\ j \in \{1{,}\ldots,8\}$$

Negative values are clamped to zero (23.9% of pairs, mainly clients far from all sites).


In [ ]:
BASE = ['employes','surface','firm_age','monthly_volume','purchase_staff']

D_mat = cdist(pot[['lat','lon']].values, sites[['lat','lon']].values)  # (200, 8)

print(f"Training distance range   : {hist['dist_retail_point'].min():.1f} – {hist['dist_retail_point'].max():.1f}")
print(f"Client–site distance range: {D_mat.min():.1f} – {D_mat.max():.1f}  (extrapolation beyond training range)")

Demand = np.zeros((200, 8))
raw_neg = 0
for j in range(8):
    raw = ols.predict(np.hstack([pot[BASE].values, D_mat[:,j].reshape(-1,1)]))
    raw_neg += (raw < 0).sum()
    Demand[:,j] = np.maximum(0, raw)

print(f"\nNegative predictions clamped: {raw_neg}/1600  ({100*raw_neg/1600:.1f}%)")
print(f"Non-zero entries            : {(Demand>0).sum()}/1600")

In [ ]:
# Build and display the demand matrix
df_demand = pd.DataFrame(
    Demand.round(2),
    index=[f"Client_{i+1}" for i in range(200)],
    columns=[f"Site_{j+1}" for j in range(8)]
)
df_demand.index.name = "Client"

print("=== DEMAND MATRIX D̂[i,j] – first 10 rows ===")
print(df_demand.head(10).to_string())
print(f"\n... ({len(df_demand)} rows total)")

print("\n=== Per-site total potential demand ===")
totals = df_demand.sum().rename("Total demand")
print(totals.round(1).to_string())
print(f"\nGrand total (if all demand captured): {df_demand.values.sum():.1f}")

In [ ]:
# Export full matrix to Excel for teammates (optimizer input)
df_demand.to_excel("demand_matrix_stage1.xlsx")
print("Full 200×8 demand matrix saved to: demand_matrix_stage1.xlsx")
print("This file is the direct input to the optimisation model.")

## 5. Summary

| Item | Value |
|---|---|
| Training data | 500 historic clients |
| Model selected | OLS Linear Regression |
| CV R² | 0.9194 (± 0.0063) |
| CV RMSE | 10.23 |
| Features used | surface, employes, firm_age, monthly_volume, purchase_staff, dist_retail_point |
| Excluded | `purchased_products` (unavailable), `loyal` (handled in Stage 2) |
| Demand matrix | 200 × 8 — output to optimisation model |
| Predictions clamped to 0 | 382 / 1600 (23.9%) |

**The 200 × 8 demand matrix `demand_matrix_stage1.xlsx` is the output of this stage**, feeding directly into the MILP optimisation model.
